## Functions

In [20]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import sys, os
from ipywidgets import interact
from IPython.display import display, Math
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display, Math
from scipy.special import genlaguerre, legendre, lpmv
from matplotlib import cm
from scipy.special import sph_harm, genlaguerre, factorial
import matplotlib as mpl
%matplotlib inline

In [21]:
def xsph(theta, phi, r):
    return np.cos(phi)*np.sin(theta)*r

def ysph(theta, phi, r):
    return np.sin(phi)*np.sin(theta)*r

def zsph(theta, phi, r):
    return np.cos(theta)*r


def az(phi,m):
    return np.exp(1j*m*phi)
def polar(theta, l, m):
    return lpmv(abs(m), l, np.cos(theta))

def ang_norm(l,m):
    num = (2*l+1)*factorial(l-abs(m))
    den = 4*np.pi*factorial(l+abs(m))
    return (num/den)**.5

def Ylm(theta, phi, l, m):
    sign = 1
    if not (m==0):
        sign = (-1)**m
    return ang_norm(l,m)*az(phi,m)*polar(theta,l,m)*sign

a0 = 5.29e-11 #m
def norm(n,l):
    return (factorial(n-l-1)/(2*n*factorial(n+l)))**.5 * (2/(n*a0))**(l+1.5)

def Rnl(r, n, l):
    gl = genlaguerre(n-l-1,2*l+1)(2*r/(n*a0))
    p1 = r**l * np.exp(-r/(n*a0))
    return norm(n,l)*gl*p1 
cmap = LinearSegmentedColormap.from_list(
    "bright_div",
    [[1,0,0], "black", [0,.3,1]]
)

In [22]:
def sph_harm():
    lwidget = widgets.IntSlider(min=0, max=4, value=1)
    mwidget = widgets.IntSlider(min=-4, max=4, value=0)
    @interact(
    elevation = widgets.FloatSlider(min=-90, max=90, step=.1,value=5),
    azim = widgets.FloatSlider(min=0, max=180, step=.1,value=30, description='rotate'),
    l = lwidget,
    m =mwidget,
    part = widgets.RadioButtons(options=['Real','Imaginary', 'Probability'],value='Real',description=' ')
    )
    def g(**kargs):
        m = kargs['m']
        l = kargs['l']

        if m > l:
            mwidget.value=l
            m=l
        if m < -l:
            mwidget.value=-l
            m=-l
        theta = np.linspace(0, np.pi, 100)
        phi = np.linspace(0, 2*np.pi, 200)

        Theta, Phi = np.meshgrid(theta, phi)
        X = xsph(Theta,Phi,1)
        Y = ysph(Theta,Phi,1)
        Z = zsph(Theta,Phi,1)

        Phi = np.atan2(Y,X)
        r = np.sqrt(X**2+Y**2+Z**2)
        Theta = np.acos(Z/r)

        if kargs['part']=='Real':
            psi = np.real(Ylm(Theta,Phi,l,m))
            norm = plt.Normalize(-psi.max(),psi.max())
            color = cmap(norm(psi))
            clabel='Re($\\psi$)'
        elif kargs['part']=='Imaginary':
            psi = np.imag(Ylm(Theta,Phi,l,m))
            if not (m==0):
                norm = plt.Normalize(-psi.max(),psi.max())
            else:
                norm = plt.Normalize(-1,1)
            color = cmap(norm(psi))
            clabel='Im($\\psi$)'

        elif kargs['part']=='Probability':
            psi = Ylm(Theta,Phi,l,m)
            psi = np.real(psi * np.conjugate(psi))
            norm = plt.Normalize(0,psi.max())
            color = cm.gray_r(norm(psi))
            clabel='($\\psi^*\\psi$)'
        
        #fig.colorbar(mappable, ax=ax, shrink=0.6)
        fig = plt.figure()
        gs = GridSpec(1,1,fig)
        ax = fig.add_subplot(gs[0],projection='3d')
        ax.plot_surface(X,Y,Z, facecolors=color, shade=False)
        ax.set_aspect('equal')
        ax.view_init(elev=kargs['elevation'], azim=kargs['azim'])
        
        mappable = mpl.cm.ScalarMappable(norm=norm, cmap=cmap if kargs['part']!='Probability' else cm.gray_r)
        mappable.set_array([])  # required for colorbar
        fig.colorbar(mappable, ax=ax, shrink=0.6,label=clabel)
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.set_title(f'$Y_{l}^{{{m}}}$')
        ax.set_axis_off()
        plt.show()



In [23]:
def lin_comb():
    lwidget = widgets.IntSlider(min=0, max=2, value=1)
    Orbital_widget = widgets.RadioButtons(options=['px','py','pz'], value='pz', description='Linear combination')

    @interact(
    elevation = widgets.FloatSlider(min=-90, max=90, step=.1,value=5),
    azim = widgets.FloatSlider(min=0, max=180, step=.1,value=30, description='Rotate'),
    l = lwidget,
    combo = Orbital_widget,
    part = widgets.RadioButtons(options=['Wavefunction', 'Probability'],description='Plot')
    )
    def g(**kargs):
        l = kargs['l']
        if l ==0:
            Orbital_widget.options=['s']
        elif l ==1:
            Orbital_widget.options=['px','py','pz']
        elif l ==2:
            Orbital_widget.options=['dz^2','dxz','dyz','dx^2-y^2','dxy']
        
        theta = np.linspace(0, np.pi, 100)
        phi = np.linspace(0, 2*np.pi, 200)

        Theta, Phi = np.meshgrid(theta, phi)
        X = xsph(Theta,Phi,1)
        Y = ysph(Theta,Phi,1)
        Z = zsph(Theta,Phi,1)

        Phi = np.atan2(Y,X)
        r = np.sqrt(X**2+Y**2+Z**2)
        Theta = np.acos(Z/r)

        combo = kargs['combo']
        if combo == 's':
            psi = Ylm(Theta,Phi,0,0)
            psi = np.real(psi)
            title = '$s= Y_0^0$'
        
        if combo=='pz':
            psi = Ylm(Theta,Phi, 1,0)
            psi = np.real(psi)
            title = '$p_z = Y_1^0$'

        if combo=='px':
            yp = Ylm(Theta, Phi, 1,1)
            ym = Ylm(Theta, Phi, 1,-1)
            psi = (ym-yp)/np.sqrt(2)
            psi = np.real(psi)
            title = '$p_x = \\dfrac{1}{\\sqrt{2}}(Y_1^{-1}+Y_1^{+1})$'

        if combo=='py':
            yp = Ylm(Theta, Phi, 1,1)
            ym = Ylm(Theta, Phi, 1,-1)
            psi = (ym+yp)/(1j * np.sqrt(2))
            psi = np.real(psi)
            title = '$p_y = \\dfrac{1}{i\\sqrt{2}}(Y_1^{-1}-Y_1^{+1})$'

        if combo=='dz^2':
            psi = Ylm(Theta,Phi,2,0)
            psi = np.real(psi)
            title = '$d_{z^2}= Y_2^0$'
        
        if combo=='dxz':
            yp = Ylm(Theta,Phi,2,1)
            ym = Ylm(Theta,Phi,2,-1)
            psi = (ym-yp)/np.sqrt(2)
            psi = np.real(psi)
            title = '$d_{xz} = \\dfrac{1}{\\sqrt{2}}(Y_2^{-1}-Y_2^{+1})$'

        if combo=='dyz':
            yp = Ylm(Theta,Phi,2,1)
            ym = Ylm(Theta,Phi,2,-1)
            psi = (ym+yp)/(1j*np.sqrt(2))
            psi = np.real(psi)
            title = '$d_{yz} = \\dfrac{1}{i\\sqrt{2}}(Y_2^{-1}+Y_2^{+1})$'
        
        if combo=='dx^2-y^2':
            yp = Ylm(Theta,Phi,2,2)
            ym = Ylm(Theta,Phi,2,-2)
            psi = (ym+yp)/(np.sqrt(2))
            psi = np.real(psi)
            title = '$d_{x^2-y^2} = \\dfrac{1}{\\sqrt{2}}(Y_2^{-2}+Y_2^{+2})$'

        if combo=='dxy':
            yp = Ylm(Theta,Phi,2,2)
            ym = Ylm(Theta,Phi,2,-2)
            psi = (ym-yp)/(1j*np.sqrt(2))
            psi = np.real(psi)
            title = '$d_{xy} = \\dfrac{1}{i\\sqrt{2}}(Y_2^{-2}-Y_2^{+2})$'

        
        
        if kargs['part']=='Wavefunction':
            norm = plt.Normalize(-psi.max(),psi.max())
            color = cmap(norm(psi))
            clabel='$\\psi$'

        elif kargs['part']=='Probability':
            psi = np.real(psi * np.conjugate(psi))
            norm = plt.Normalize(0,psi.max())
            color = cm.gray_r(norm(psi))
            clabel='$\\psi^*\\psi$'


        fig = plt.figure()

        ax = fig.add_subplot(projection='3d')
        ax.plot_surface(X,Y,Z, facecolors=color, shade=False)
        ax.set_aspect('equal')
        ax.view_init(elev=kargs['elevation'], azim=kargs['azim'])
        
        mappable = mpl.cm.ScalarMappable(norm=norm, cmap=cmap if kargs['part']!='Probability' else cm.gray_r)
        mappable.set_array([])  # required for colorbar
        fig.colorbar(mappable, ax=ax, shrink=0.6,label=clabel)
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.set_title(title)
        ax.set_axis_off()
        plt.show()


In [ ]:
def hatom_ylm():
    lwidget = widgets.IntSlider(min=0, max=3, value=1)
    mwidget = widgets.IntSlider(min=-3, max=3, value=0)
    @interact(
        #tranparency= widgets.FloatSlider(min=.01, max=.5, value=.1, step=.001),
        part = widgets.RadioButtons(options=['Complex','Real', 'Imaginary'], value='Real', description='Real or Imaginary'),
        Xcheck = widgets.Checkbox(description='Split X'),
        Ycheck = widgets.Checkbox(description='Split Y'),
        Zcheck = widgets.Checkbox(description='Split Z'),
        scalebar=widgets.ToggleButton(value=True),
        prob = widgets.RadioButtons(options=['Wavefunction', 'Probability Density'], value='Wavefunction',description='Wavefunction or Prob.'),
        n = widgets.IntSlider(min=1, max=4, value=2),
        l = lwidget,
        m = mwidget,
        elevation = widgets.FloatSlider(min=-90, max=90, value=5, step=1),
        rotate = widgets.FloatSlider(min=0, max=360, value=235, step=1),
    )
    def g(**kargs):
        kargs['tranparency'] = .6
        lwidget.max= kargs['n']-1
        if kargs['l'] > kargs['n']-1:
            lwidget.value = kargs['n'] - 1
            return 'l < n-1'
        
        mwidget.min=-kargs['l']
        mwidget.max=kargs['l']
        if kargs['m'] > kargs['l']:
            mwidget.value=kargs['l']
            return 'm <= l'
        if kargs['m'] < -kargs['l']:
            mwidget.value=-kargs['l']
            return 'm >= -l'
        scale = int(kargs['n']**2 * 4)
        extent = scale*a0
        x = np.linspace(-extent, extent, 50)
        X,Y,Z = np.meshgrid(x,x,x)

        Phi = np.atan2(Y,X)
        R = (X**2+Y**2+Z**2)**.5
        Theta = np.arccos(Z/R)

        fig = plt.figure()
        ax = fig.add_subplot(projection='3d')
        n=kargs['n']
        l = kargs['l']
        m=kargs['m']
        psi = Ylm(Theta,Phi,l,m) * Rnl(R,n,l)
        #dP = np.real(psi*np.conj(psi))
        mx = np.max(np.real(psi))
        inds = np.argwhere(abs(psi) > .05 * mx)

        X_plot = X[inds[:,0], inds[:,1], inds[:,2]].ravel()
        Y_plot = Y[inds[:,0], inds[:,1], inds[:,2]].ravel()
        Z_plot = Z[inds[:,0], inds[:,1], inds[:,2]].ravel()
        psi_plot = psi[inds[:,0], inds[:,1], inds[:,2]].ravel()


        
        if kargs['Xcheck']:
            xsplit = np.argwhere(X_plot > 0)[:, -1]
        else:
            xsplit = np.argwhere(X_plot > -np.inf)[:, -1]
        if kargs['Ycheck']:
            ysplit = np.argwhere(Y_plot[xsplit] > 0)[:, -1]
        else:
            ysplit = np.argwhere(Y_plot[xsplit] > -np.inf)[:, -1]
        if kargs['Zcheck']:
            zsplit = np.argwhere(Z_plot[xsplit][ysplit] > 0)[:, -1]
        else:
            zsplit = np.argwhere(Z_plot[xsplit][ysplit] > -np.inf)[:, -1]
        X_plot = X_plot[xsplit][ysplit][zsplit]
        Y_plot = Y_plot[xsplit][ysplit][zsplit]
        Z_plot = Z_plot[xsplit][ysplit][zsplit]
        psi_plot = psi_plot[xsplit][ysplit][zsplit]



        if kargs['prob']=='Probability Density':
            psi_plot = np.real(psi_plot * np.conj(psi_plot))
            colors = np.zeros((len(psi_plot), 4))  
            alpha = psi_plot/np.max(psi_plot)
            colors[:, 3] = alpha
        elif kargs['prob']=='Wavefunction':
            if kargs['part']=='Real':
                psi_plot = np.real(psi_plot)
                reds = np.zeros_like(psi_plot)
                reds[np.argwhere(psi_plot>0)] = .8
                blues = np.zeros_like(psi_plot)
                blues[np.argwhere(psi_plot<=0)] = .8
                alpha = abs(psi_plot)/np.max(abs(psi_plot))*kargs['tranparency']
                colors = np.zeros((len(psi_plot), 4))  
                colors[:, 0] = reds
                colors[:, 2] = blues
                colors[:, 3] = alpha
            elif kargs['part']=='Imaginary':
                psi_plot = np.imag(psi_plot)
                reds = np.zeros_like(psi_plot)
                reds[np.argwhere(psi_plot>0)] = .8
                blues = np.zeros_like(psi_plot)
                blues[np.argwhere(psi_plot<=0)] = .8
                alpha = abs(psi_plot)/np.max(abs(psi_plot))*kargs['tranparency']
                colors = np.zeros((len(psi_plot), 4))  
                colors[:, 0] = reds
                colors[:, 2] = blues
                colors[:, 3] = alpha
            elif kargs['part']== 'Complex':
                angles = np.angle(psi_plot)+np.pi/4
                norm = plt.Normalize(-np.pi, np.pi)
                colors = cm.hsv(norm(angles))
                alpha = abs(psi_plot)/np.max(abs(psi_plot))*kargs['tranparency']
                colors[:, 3] = alpha

    #c=colors[order], s=1, rasterized=True)

        ax.scatter(X_plot, Y_plot, Z_plot, c=colors, s=10,rasterized=True)
        scalebar = np.array([0, a0])*4
        if kargs['scalebar']:
            ax.text(-scale*a0, 0, 0, '$4a_0$')
            ax.plot(-scale*a0 +scalebar, [0,0], [0,0], 'k')
        ax.set_xlim(-extent, extent)
        ax.set_ylim(-extent, extent)
        ax.set_zlim(-extent, extent)
        ax.set_aspect('equal')
        ax.view_init(elev=kargs['elevation'], azim=-kargs['rotate'])
        ax.set_axis_off()
        #ax.set_xlabel('x')
        #ax.set_ylabel('y')
        plt.show()
hatom_ylm()

interactive(children=(RadioButtons(description='Real or Imaginary', index=1, options=('Complex', 'Real', 'Imag…

In [25]:
def hatom_comb():
    lwidget = widgets.IntSlider(min=0, max=2, value=1)
    Orbital_widget = widgets.RadioButtons(options=['px','py','pz'], value='pz', description='Linear combination')
    savewidget= widgets.ToggleButton(description='Save')
    @interact(
    #tranparency= widgets.FloatSlider(min=.01, max=.5, value=.3, step=.001),
    n = widgets.IntSlider(min=1, max=4, value=2),
    l = lwidget,
    combo = Orbital_widget,
    prob = widgets.RadioButtons(options=['Wavefunction', 'Probability Density'], value='Wavefunction'),
    Xcheck = widgets.Checkbox(description='Split X'),
    Ycheck = widgets.Checkbox(description='Split Y'),
    Zcheck = widgets.Checkbox(description='Split Z'),
    elevation = widgets.FloatSlider(min=-90, max=90, step=.1,value=0),
    azim = widgets.FloatSlider(min=0, max=360, step=.1,value=135, description='rotate'),
    )
    def g(**kargs):
        if kargs['n']==1:
            lwidget.max= 0
        if kargs['n']==2:
            lwidget.max = 1
        if kargs['n']==3 or kargs['n']==4:
            lwidget.max=2
        if kargs['l'] > kargs['n']-1:
            lwidget.value = kargs['n'] - 1
            return 'l < n-1'
        n = kargs['n']
        l = kargs['l']
        kargs['tranparency']=.3
        p = 0.05
        if l > n-1:
            lwidget.value=n-1
            l = n-1
            return 'l < n-1'
        if l ==0:
            Orbital_widget.options=['s']
        elif l ==1:
            Orbital_widget.options=['px','py','pz']
        elif l ==2:
            Orbital_widget.options=['dz^2','dxz','dyz','dx^2-y^2','dxy']
        scale = int(kargs['n']**2 * 4)
        extent = scale*a0
        x = np.linspace(-extent, extent, 50)
        X,Y,Z = np.meshgrid(x,x,x)

        Phi = np.atan2(Y,X)
        R = (X**2+Y**2+Z**2)**.5
        Theta = np.arccos(Z/R)

        combo = kargs['combo']
        if combo == 's':
            psi = Ylm(Theta,Phi,0,0)*Rnl(R,n,l)
            psi = np.real(psi)
            title = '$Y_0^0$'
            title = f'${n}s = R_{{{n},{l}}}$' + title
        
        if combo=='pz':
            psi = Ylm(Theta,Phi, 1,0)*Rnl(R,n,l)
            psi = np.real(psi)
            title = '$Y_1^0$'
            title = f'${n}p_z=R_{{{n},{l}}}$'+title

        if combo=='px':
            yp = Ylm(Theta, Phi, 1,1)*Rnl(R,n,l)
            ym = Ylm(Theta, Phi, 1,-1)*Rnl(R,n,l)
            psi = (ym+yp)/np.sqrt(2)
            psi = np.real(psi)
            title = '$\\dfrac{1}{\\sqrt{2}}(Y_1^{-1}+Y_1^{+1})$'
            title = f'${n}p_x = R_{{{n},{l}}}$'+title

        if combo=='py':
            yp = Ylm(Theta, Phi, 1,1)*Rnl(R,n,l)
            ym = Ylm(Theta, Phi, 1,-1)*Rnl(R,n,l)
            psi = (ym-yp)/(1j * np.sqrt(2))
            psi = np.real(psi)
            title = '$\\dfrac{1}{i\\sqrt{2}}(Y_1^{-1}-Y_1^{+1})$'
            title = f'${n}p_y = R_{{{n},{l}}}$'+title

        if combo=='dz^2':
            psi = Ylm(Theta,Phi,2,0)*Rnl(R,n,l)
            psi = np.real(psi)
            title = '$Y_2^0$'
            title = f'${n}d_{{z^2}} = R_{{{n},{l}}}$'+title
        
        if combo=='dxz':
            yp = Ylm(Theta,Phi,2,1)*Rnl(R,n,l)
            ym = Ylm(Theta,Phi,2,-1)*Rnl(R,n,l)
            psi = (ym+yp)/np.sqrt(2)
            psi = np.real(psi)
            title = '$\\dfrac{1}{\\sqrt{2}}(Y_2^{-1}+Y_2^{+1})$'
            title = f'${n}d_{{xz}}= R_{{{n},{l}}}$'+title

        if combo=='dyz':
            yp = Ylm(Theta,Phi,2,1)*Rnl(R,n,l)
            ym = Ylm(Theta,Phi,2,-1)*Rnl(R,n,l)
            psi = (ym-yp)/(1j*np.sqrt(2))
            psi = np.real(psi)
            title = '$\\dfrac{1}{i\\sqrt{2}}(Y_2^{-1}-Y_2^{+1})$'
            title = f'${n}d_{{yz}}=R_{{{n},{l}}}$'+title
        
        if combo=='dx^2-y^2':
            yp = Ylm(Theta,Phi,2,2)*Rnl(R,n,l)
            ym = Ylm(Theta,Phi,2,-2)*Rnl(R,n,l)
            psi = (ym+yp)/(np.sqrt(2))
            psi = np.real(psi)
            title = '$\\dfrac{1}{\\sqrt{2}}(Y_2^{-2}+Y_2^{+2})$'
            title = f'${n}d_{{x^2-y^2}}=R_{{{n},{l}}}$'+title

        if combo=='dxy':
            yp = Ylm(Theta,Phi,2,2)*Rnl(R,n,l)
            ym = Ylm(Theta,Phi,2,-2)*Rnl(R,n,l)
            psi = (ym-yp)/(1j*np.sqrt(2))
            psi = np.real(psi)
            title = '$\\dfrac{1}{i\\sqrt{2}}(Y_2^{-2}-Y_2^{+2})$'
            title = f'${n}d_{{xy}}=R_{{{n},{l}}}$'+title

        

        mx = np.max(np.real(psi))
        inds = np.argwhere(abs(psi) >p* mx)

        X_plot = X[inds[:,0], inds[:,1], inds[:,2]].ravel()
        Y_plot = Y[inds[:,0], inds[:,1], inds[:,2]].ravel()
        Z_plot = Z[inds[:,0], inds[:,1], inds[:,2]].ravel()
        psi_plot = psi[inds[:,0], inds[:,1], inds[:,2]].ravel()


        if kargs['Xcheck']:
            xsplit = np.argwhere(X_plot > 0)[:, -1]
        else:
            xsplit = np.argwhere(X_plot > -np.inf)[:, -1]
        if kargs['Ycheck']:
            ysplit = np.argwhere(Y_plot[xsplit] > 0)[:, -1]
        else:
            ysplit = np.argwhere(Y_plot[xsplit] > -np.inf)[:, -1]
        if kargs['Zcheck']:
            zsplit = np.argwhere(Z_plot[xsplit][ysplit] > 0)[:, -1]
        else:
            zsplit = np.argwhere(Z_plot[xsplit][ysplit] > -np.inf)[:, -1]
        X_plot = X_plot[xsplit][ysplit][zsplit]
        Y_plot = Y_plot[xsplit][ysplit][zsplit]
        Z_plot = Z_plot[xsplit][ysplit][zsplit]
        psi_plot = psi_plot[xsplit][ysplit][zsplit]
        
        if kargs['prob']=='Probability Density':
            psi_plot = np.real(psi_plot * np.conj(psi_plot))
            colors = np.zeros((len(psi_plot), 4))  
            alpha = psi_plot/np.max(psi_plot)*kargs['tranparency']*2
            colors[:, 3] = alpha
        elif kargs['prob']=='Wavefunction':
            psi_plot = np.real(psi_plot)
            reds = np.zeros_like(psi_plot)
            reds[np.argwhere(psi_plot>0)] = .8
            blues = np.zeros_like(psi_plot)
            blues[np.argwhere(psi_plot<=0)] = .8
            alpha = abs(psi_plot)/np.max(abs(psi_plot))*kargs['tranparency']
            colors = np.zeros((len(psi_plot), 4))  
            colors[:, 0] = reds
            colors[:, 2] = blues
            colors[:, 3] = alpha


        fig = plt.figure()

        ax = fig.add_subplot(projection='3d')
        #ax.plot_surface(X,Y,Z, facecolors=color, shade=False)
        ax.set_aspect('equal')
        ax.view_init(elev=kargs['elevation'], azim=-kargs['azim'])
        order = np.argsort(Z_plot)
        ax.scatter(X_plot[order], Y_plot[order], Z_plot[order], 
                c=colors[order], s=10, rasterized=True)
        
        scalebar = np.array([0, a0])*4
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.set_title(title)
        ax.set_axis_off()
        ax.set_xlim(-extent, extent)
        ax.set_ylim(-extent, extent)
        ax.set_zlim(-extent, extent)
        
    
        plt.show()
hatom_comb()

interactive(children=(IntSlider(value=2, description='n', max=4, min=1), IntSlider(value=1, description='l', m…

In [26]:
def radial():
    lwidget = widgets.IntSlider(min=0, max=5, step=1, value=0)
    savewidget = widgets.ToggleButton(value=False, description='Save')
    @interact(
    n = widgets.IntSlider(min=1, max=5, step=1, value=2),
    l = lwidget,
    prob = widgets.ToggleButton(description='Probability')
    )
    def g(**kargs):
        if kargs['l'] > kargs['n']-1:
            lwidget.value=kargs['n']-1
            l = kargs['n']-1
        else:
            l=kargs['l']
        r = np.linspace(0, 40*a0, 200)
        fig = plt.figure(figsize=(4,3))
        gs= GridSpec(1,1,fig)
        axs = []
        axs.append(fig.add_subplot(gs[0]))

        ax = axs[0]
        r = np.linspace(0, 30*a0, 200)
        if (kargs['prob']):
            ax.plot(r*1e10, Rnl(r, kargs['n'], l)**2*r**2)
            ax.axhline(0, color='k', linestyle='--')
            ax.set_xlim(0, r[-1]*1e10)
            ax.set_xlabel('r($\\AA$)')
            ax.set_ylabel('$r^2 R(r)^2$ (m$^{-3}$)')
            #ax.set_ylim(0,1.2e10)
        if (not kargs['prob']):
            ax.plot(r*1e10, Rnl(r, kargs['n'], l))
            ax.axhline(0, color='k', linestyle='--')
            ax.set_xlim(0, r[-1]*1e10)
            ax.set_xlabel('r($\\AA$)')
            ax.set_ylabel('$R(r)$ (m$^{-3/2}$)')
            #ax.set_ylim(-1e15,5e15)

        plt.show()


## Radial

In [27]:
radial()

interactive(children=(IntSlider(value=2, description='n', max=5, min=1), IntSlider(value=0, description='l', m…

## Spherical Harmonics 

In [28]:
sph_harm()

interactive(children=(FloatSlider(value=5.0, description='elevation', max=90.0, min=-90.0), FloatSlider(value=…

## Linear Combinations

In [29]:
lin_comb()

interactive(children=(FloatSlider(value=5.0, description='elevation', max=90.0, min=-90.0), FloatSlider(value=…

## H atom wavefunctions $\psi_{n,l,m}$

In [ ]:
hatom_ylm()

interactive(children=(RadioButtons(description='Real or Imaginary', index=1, options=('Complex', 'Real', 'Imag…

## Linear Combinations of H atom wavefunctions $\psi_{n,l,m}$

In [31]:
hatom_comb()

interactive(children=(IntSlider(value=2, description='n', max=4, min=1), IntSlider(value=1, description='l', m…